# Guided Group Featurizers: reproduce and check

Use **Runtime → Change runtime type → GPU**, then **Run all**. Upload the repository
ZIP when prompted and enable the `HF_TOKEN` Colab secret for the gated DINOv3 model.

The default **quick** mode checks each software stage with small workloads:
public COCO/VOC inputs, DINOv3 extraction, GGF and baseline updates, category-split
training, strict SLiCS solving, all three Concept-SAE stages, concentration scoring,
synthetic recovery, geometry and transformations, execution profiling,
historical fit and continuation kernels, review-card generation and figure rendering. It also recounts the supplied paper
results. Expect public data and model downloads on the first run.

Quick outputs use a separate directory and are integration tests, not paper results.
The full mode exposes the recorded experiment stages. Fresh visual judgments need
reviewer access and completed labels; this notebook does not submit paid requests.

In [ ]:
RUN_MODE = "quick"  # "quick", "evidence" or "full"
RUN_NAME = "ggf_quick_check"  # Keep this name to resume the same source edition.

## 1. Locate the repository

In [ ]:
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
import json
import subprocess
import sys


def locate_repository():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "ggf_repository"]
    for candidate in candidates:
        if (candidate / "code/workflows.py").is_file():
            return candidate.resolve()
    archives = [
        p for pattern in ("ggf_anonymous_repository*.zip", "GGF_CVPR2027_supplementary*.zip")
        for p in Path.cwd().glob(pattern)
    ]
    if not archives:
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError("Extract the repository ZIP and run the notebook from its root.") from error
        print("Upload ggf_anonymous_repository.zip or GGF_CVPR2027_supplementary.zip.")
        uploaded = files.upload()
        archives = [Path(name) for name in uploaded if name.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Provide exactly one repository ZIP in the working directory.")
    destination = Path.cwd() / "ggf_repository"
    with ZipFile(archives[0]) as archive:
        manifest = json.loads(archive.read("FILES.json"))
        if set(archive.namelist()) != set(manifest) | {"FILES.json"}:
            raise ValueError("ZIP inventory differs from FILES.json")
        payload = {}
        for name in archive.namelist():
            path = destination / name
            if ":" in name or "\\" in name or not path.resolve().is_relative_to(destination.resolve()):
                raise ValueError("Invalid ZIP member path")
            raw = archive.read(name)
            if name != "FILES.json" and sha256(raw).hexdigest() != manifest[name]:
                raise ValueError("ZIP file checksum mismatch: " + name)
            if path.exists() and path.read_bytes() != raw:
                raise FileExistsError("Existing extracted file differs: " + name)
            payload[path] = raw
        for path, raw in payload.items():
            path.parent.mkdir(parents=True, exist_ok=True)
            if not path.exists():
                path.write_bytes(raw)
    if not (destination / "code/workflows.py").is_file():
        raise FileNotFoundError("The ZIP does not contain the reproduction repository.")
    return destination.resolve()


REPO = locate_repository()
CODE = REPO / "code"
print("Repository:", REPO)

sys.path.insert(0, str(CODE / "bootstrap"))
from command_runner import run_command
LOGS = REPO.parent / "ggf_logs"


from datetime import datetime, timezone

if RUN_MODE not in {"quick", "evidence", "full"}:
    raise ValueError("Choose quick, evidence or full.")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME must be a directory name.")
RUN = REPO.parent / RUN_NAME
print("Mode:", RUN_MODE, "Output:", RUN)

## 2. Install and check the environment

The GPU modes install the recorded CUDA 13 Torch-family wheels and numerical
dependencies. The image-import check runs in a child process to detect binary
incompatibilities before downloading the model. Evidence-only mode uses Python's
standard library and skips these installations.

In [ ]:
if RUN_MODE != "evidence":
    import textwrap
    expected = json.loads((CODE / "bootstrap/SPECIFICATION.json").read_text())["numerical_versions"]
    if expected["torch"] != "2.11.0+cu130":
        raise ValueError("Review the matched torchvision wheels before changing the recorded Torch version.")
    run_command([sys.executable, "-m", "pip", "install",
                 "torch==2.11.0+cu130", "torchvision==0.26.0+cu130", "torchaudio==2.11.0+cu130",
                 "--index-url", "https://download.pytorch.org/whl/cu130"], LOGS, "install_torch_family")
    run_command([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt"),
                 *[package + "==" + version for package, version in expected.items() if package != "torch"]], LOGS, "install_dependencies")
    image_import_check = """
    import torch, torchvision, torchaudio
    from torchvision.ops import nms
    from transformers import AutoImageProcessor, AutoModel
    assert str(torch.__version__) == '2.11.0+cu130', torch.__version__
    assert str(torchvision.__version__) == '0.26.0+cu130', torchvision.__version__
    assert str(torchaudio.__version__) == '2.11.0+cu130', torchaudio.__version__
    keep = nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([1.]), 0.5)
    assert keep.tolist() == [0]
    print('Torch, torchvision operators and Transformers image/model imports passed.')
    """
    run_command([sys.executable, "-c", textwrap.dedent(image_import_check)], LOGS, "check_image_imports")

## 3. Model access

In [ ]:
if RUN_MODE != "evidence":
    import os

    if not os.environ.get("HF_TOKEN"):
        try:
            from google.colab import userdata
        except ImportError:
            raise RuntimeError("Set the HF_TOKEN environment variable before running the model.")
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 4. Verify the supplied evidence

This checks the archive hashes, cited Appendix paths, saved review records,
GPT and Gemini recounts, and numerical summaries. It can also run by itself with
`RUN_MODE = "evidence"`.

In [ ]:
run_command([sys.executable, str(REPO / "scripts/verify_bundle.py")], LOGS, "verify_evidence")

## 5. Quick pipeline check

Each section runs in its own Python process and writes a receipt after success.
Rerunning reuses completed sections from the same source edition. Failed commands
show their output and save a full log. `QUICK_COMPLETE.json` is written only after
every section passes.

The optimizer and scoring checks use small fixtures with known labels. The public
input checks use real images and pinned DINOv3 weights. Quick card generation checks the six-view layout; semantic review is a separate task.
The VOC check reads four images, but verifying the official archive requires its
full download. Keep the runtime connected during downloads.

In [ ]:
if RUN_MODE == "quick":
    run_command([sys.executable, str(CODE / "bootstrap/quick_pipeline.py"),
                 "--code-root", str(CODE), "--root", str(RUN), "--device", "cuda"],
                LOGS, "quick_pipeline")
    receipt = json.loads((RUN / "QUICK_COMPLETE.json").read_bytes())
    if not receipt.get("complete") or receipt.get("research_run") is not False:
        raise ValueError("The complete quick-check receipt is missing.")
    print("Passed sections:", ", ".join(receipt["sections"]))
    print("Quick check complete:", RUN / "QUICK_COMPLETE.json")

## 6. Full experiment stages

Use a separate `RUN_NAME`, for example `ggf_full_reproduction`, with `RUN_MODE = "full"`.
The planner resolves prerequisites automatically. It covers the matched 75-group
study, historical width and continuation controls, COCO/VOC comparisons,
Concept-SAE, synthetic recovery, frozen-model analyses and execution profiles.
Every study creates its inputs and checkpoints under this run directory.
Full research budgets take much longer than the quick check.

First inspect the printed plan with `EXECUTE_FULL = False`. Set it to `True` to run.
An empty `FULL_STAGES` selects all numerical studies and prepares blinded cards.
To run a subset, use names such as `("controls.coco.evaluate",)`; prerequisites
are included. Add `INCLUDE_ANNOTATIONS = True` after supplying fresh pair/whole
reviews and presence labels with independent QA. See `docs/REPRODUCING.md` for
the label paths, access requirements and validation status.
Feature normalization uses disk-backed arrays. If extraction stopped in a previous
source edition, use a new `RUN_NAME` and set `FEATURE_CACHE` to its `matched75`
directory. Compatible, checksum-verified development shards are copied into the
new run; fitted models and labels are not imported. Keep that previous directory
available until extraction completes.


In [ ]:
FULL_STAGES = ()  # Example: ("controls.coco.evaluate",)
EXECUTE_FULL = False
INCLUDE_ANNOTATIONS = False
FEATURE_CACHE = ""  # Optional previous matched75 run directory for verified feature shards.

if RUN_MODE == "full":
    command = [sys.executable, str(CODE / "bootstrap/full_pipeline.py"),
               "--code-root", str(CODE), "--root", str(RUN)]
    if FULL_STAGES:
        command += ["--stages", *FULL_STAGES]
    if FEATURE_CACHE:
        command += ["--feature-cache", FEATURE_CACHE]
    if INCLUDE_ANNOTATIONS:
        command += ["--include-annotations"]
    if EXECUTE_FULL:
        command += ["--execute"]
    run_command(command, LOGS, "full_pipeline")

## 7. Figures and supporting studies

Quick mode already renders the four supplied figure families. In full mode, the
cell below lists the available study workflows and optionally renders those figures.
`docs/FRESH_SYNTHETIC.md` and `docs/FRESH_VOC.md` give the synthetic and VOC commands.
The full planner rebuilds historical prerequisites from public inputs. Original
source editions are supplied separately for inspecting the recorded algorithms.
The quick check validates small workloads, not full research-budget execution.

In [ ]:
REBUILD_FIGURES = False
if RUN_MODE == "full":
    run_command([sys.executable, str(CODE / "workflows.py"), "list"], LOGS, "list_workflows")
    if REBUILD_FIGURES:
        for name in ("geometry", "architecture", "interpretability", "voc"):
            run_command([sys.executable, str(CODE / "workflows.py"), "run",
                         "figures." + name, "--execute"], LOGS, "figures_" + name)